# DTM Drainage AI — Exploration Notebook
### MoPR Geospatial Intelligence Hackathon

This notebook walks through the full pipeline interactively for the Gujarat DEVDI dataset.

In [ ]:
import sys
sys.path.insert(0, '..')  # add repo root

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import rasterio
from rasterio.plot import show
import geopandas as gpd

# Repo modules
from src.preprocessing.point_cloud_loader import inspect, load_chunked
from src.dtm.dtm_generator import get_dtm_stats

plt.rcParams['figure.dpi'] = 150
print('Setup complete ✓')

## 1. Data Inspection

In [ ]:
LAS_PATH = '../data/input/DEVDI_POINT_CLOUD_511671.las'

meta = inspect(LAS_PATH)
print(meta)

print(f'\n⚠  Known data challenges:')
print(f'   Intensity range: {meta.intensity_range}  →  Zero intensity = geometry-only features')
print(f'   Classification: {meta.has_classification}  →  SMRF ground classification needed')

In [ ]:
# Sample 200,000 points for visualization
import laspy
chunk = next(load_chunked(LAS_PATH, chunk_size=200_000))

x = np.array(chunk.x)
y = np.array(chunk.y)
z = np.array(chunk.z)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top-view elevation scatter
sc = axes[0].scatter(x, y, c=z, cmap='terrain', s=0.5, alpha=0.6)
plt.colorbar(sc, ax=axes[0], label='Elevation (m)')
axes[0].set_title('DEVDI — Top View (elevation coloured)', fontsize=12)
axes[0].set_xlabel('Easting (m)')
axes[0].set_ylabel('Northing (m)')
axes[0].set_aspect('equal')

# Elevation histogram
axes[1].hist(z, bins=100, color='steelblue', edgecolor='none', alpha=0.8)
axes[1].set_xlabel('Elevation (m)')
axes[1].set_ylabel('Point count')
axes[1].set_title('Elevation Distribution (200k sample)', fontsize=12)

plt.tight_layout()
plt.savefig('../data/output/point_cloud_overview.png', bbox_inches='tight')
plt.show()

## 2. DTM Visualization

In [ ]:
DTM_PATH = '../data/output/devdi/dtm.tif'

stats = get_dtm_stats(DTM_PATH)
print('DTM Statistics:')
for k, v in stats.items():
    print(f'  {k:25s}: {v}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

layers = [
    ('dtm.tif',                    'terrain',    'DTM Elevation'),
    ('slope.tif',                  'hot_r',      'Slope (degrees)'),
    ('twi.tif',                    'Blues',      'TWI (Wetness Index)'),
    ('flow_accumulation.tif',      'YlOrRd',     'Flow Accumulation (log)'),
    ('waterlogging_probability.tif','RdYlGn_r',  'Waterlogging Probability'),
    ('hillshade.tif',              'gray',       'Hillshade'),
]

import os
out_dir = '../data/output/devdi/'

for ax, (fname, cmap, title) in zip(axes.flat, layers):
    fpath = os.path.join(out_dir, fname)
    if os.path.exists(fpath):
        with rasterio.open(fpath) as src:
            data = src.read(1, masked=True)
        im = ax.imshow(data, cmap=cmap, interpolation='nearest')
        plt.colorbar(im, ax=ax, fraction=0.046)
    else:
        ax.text(0.5, 0.5, f'{fname}\nnot yet generated', 
                ha='center', va='center', transform=ax.transAxes)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.axis('off')

plt.suptitle('DEVDI Village — Terrain & Hydrology Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/output/terrain_analysis_dashboard.png', bbox_inches='tight', dpi=200)
plt.show()

## 3. Drainage Network Visualization

In [ ]:
import geopandas as gpd

GPKG = '../data/output/devdi/drainage_network.gpkg'

try:
    channels   = gpd.read_file(GPKG, layer='drainage_channels')
    hotspots   = gpd.read_file(GPKG, layer='waterlogging_hotspots')
    depressions = gpd.read_file(GPKG, layer='depression_polygons')
    
    print(f'Channels  : {len(channels)} segments')
    print(f'Hotspots  : {len(hotspots)} polygons')
    print(f'Depressions: {len(depressions)} polygons')
    
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # Background: DTM hillshade
    hs_path = '../data/output/devdi/hillshade.tif'
    if os.path.exists(hs_path):
        with rasterio.open(hs_path) as src:
            hs = src.read(1, masked=True)
            ext = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]
        ax.imshow(hs, extent=ext, cmap='gray', alpha=0.5, origin='upper')
    
    # Waterlogging hotspots
    color_map = {'LOW': 'yellow', 'MEDIUM': 'orange', 'HIGH': 'red'}
    for risk, grp in hotspots.groupby('risk_level'):
        grp.plot(ax=ax, color=color_map.get(risk, 'gray'), alpha=0.6, label=f'{risk} risk')
    
    # Drainage channels coloured by channel type
    if 'channel_type' in channels.columns:
        for ctype, grp in channels.groupby('channel_type'):
            c = 'blue' if ctype == 'earthen' else 'navy'
            grp.plot(ax=ax, color=c, linewidth=1.5, label=f'{ctype} channel')
    else:
        channels.plot(ax=ax, color='blue', linewidth=1.5, label='channels')
    
    ax.legend(loc='upper right', fontsize=9)
    ax.set_title('DEVDI — Drainage Network Design & Waterlogging Zones', 
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Easting (m)')
    ax.set_ylabel('Northing (m)')
    
    plt.tight_layout()
    plt.savefig('../data/output/drainage_design_map.png', bbox_inches='tight', dpi=200)
    plt.show()
    
except Exception as e:
    print(f'Run the full pipeline first: {e}')

## 4. XGBoost Feature Importance

In [ ]:
import joblib
import pandas as pd

model_path = '../data/output/devdi/models/waterlogging_xgb.joblib'

try:
    data = joblib.load(model_path)
    model = data['model']
    
    feature_names = [
        'Elevation (norm)', 'Slope', 'Aspect', 'TWI', 'TPI',
        'Log Flow Acc.', 'Plan Curv.', 'Profile Curv.',
        'Depression Depth', 'Dist to Stream'
    ]
    
    fi = pd.Series(model.feature_importances_, index=feature_names[:len(model.feature_importances_)])
    fi = fi.sort_values(ascending=True)
    
    fig, ax = plt.subplots(figsize=(8, 5))
    fi.plot(kind='barh', ax=ax, color='steelblue', edgecolor='none')
    ax.set_xlabel('Feature Importance (XGBoost gain)')
    ax.set_title('Waterlogging Model — Feature Importances', fontweight='bold')
    ax.axvline(fi.mean(), color='red', linestyle='--', alpha=0.7, label='mean')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../data/output/feature_importance.png', bbox_inches='tight')
    plt.show()
    
except Exception as e:
    print(f'Train the model first: {e}')

## 5. Hydraulic Design Summary

In [ ]:
try:
    channels_df = gpd.read_file(GPKG, layer='drainage_channels')
    
    print('═' * 60)
    print('DRAINAGE NETWORK DESIGN SUMMARY')
    print('═' * 60)
    print(f'Total segments    : {len(channels_df)}')
    print(f'Total length      : {channels_df["length_m"].sum():,.0f} m')
    if 'cost_inr' in channels_df.columns:
        print(f'Total cost (est.) : ₹{channels_df["cost_inr"].sum():,.0f}')
    if 'channel_type' in channels_df.columns:
        print(f'Channel types     : {channels_df["channel_type"].value_counts().to_dict()}')
    if 'depth_m' in channels_df.columns:
        print(f'Depth range       : {channels_df["depth_m"].min():.2f} – {channels_df["depth_m"].max():.2f} m')
    if 'Q_design_m3s' in channels_df.columns:
        print(f'Peak discharge    : {channels_df["Q_design_m3s"].max():.4f} m³/s')
    print('═' * 60)
    
    display(channels_df[['segment_id', 'length_m', 'channel_type', 
                          'bottom_width_m', 'depth_m', 'Q_design_m3s', 
                          'cost_inr']].head(10))
except Exception as e:
    print(f'Run the pipeline first: {e}')